# 06 — Hybrid Construction, Decision Freeze, and Final Policy Comparison

This notebook compares every TraceAdvisor recommendation policy for the Google subset through the shared evaluator.

## Pipeline position

- **N04** — corrected occurrence artifacts and baseline recommendations.
- **N05** — tuned and locked ML models; final ML predictions and P95 inputs.
- **N06 (this notebook)** — evaluate Request/Mean/P90/P95/P99, receive the ML predictions, construct Hybrid recommendations, and compare all policies through the shared evaluator.

## Responsibilities

1. Load the N04 occurrence artifacts and the N05 predictions and metadata; verify identifiers, target alignment, and the P95 inputs.
2. Construct Hybrid recommendations with the shared rule in `src/policies/hybrid.py`: `max(ML prediction, P95 baseline)`, using the ML prediction where P95 is unavailable.
3. Compare Request, Mean, P90, P95, P99, ML-only, and Hybrid on the common validation population. Hybrid with the P99 safeguard is reported as a supplementary variant.
4. Freeze every decision in a decision manifest that records the decisions and a SHA-256 hash of every input artifact (backlog Issue 11).
5. Run the one-time final-test evaluation only when explicitly enabled, and only if the decision manifest exists and every input hash still matches (backlog Issue 12).

### Final-test discipline

The final-test section is disabled by default (`RUN_FINAL_TEST = False`). The test targets are not read anywhere else in this notebook.

### Status of these results: preliminary

Under the team plan (`docs/planning/data_integration_tasks.md`, Task 9), Google targets are currently built before split assignment, so 104 of the 329 common validation occurrences have targets that fall in the test period, and "any Google result produced before Task 9" is preliminary. Therefore:

- every validation number in this notebook is **preliminary**;
- the decision manifest written here is marked `"status": "preliminary"`, and the final-test section refuses to run unless the manifest status is `"final"`;
- freezing and testing are separate runs: run once with `FREEZE_STATUS = "final"` (on a clean, committed tree) to write the final manifest, then run once with `RUN_FINAL_TEST = True`, which reads that existing manifest and never rewrites it;
- the one-time Google final test runs only after Task 9 corrects the targets and the team approves the frozen decisions.

Nothing in this notebook is tuned or selected: the baseline policy and `min_runs` come from N04, the ML models from N05, and the Hybrid rule is fixed in advance.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import subprocess
import sys

import numpy as np
import pandas as pd

In [2]:
# Locate repository root.
REPO_ROOT = Path.cwd()

while (
    REPO_ROOT.name != "masters-project"
    and REPO_ROOT.parent != REPO_ROOT
):
    REPO_ROOT = REPO_ROOT.parent

if REPO_ROOT.name != "masters-project":
    raise RuntimeError(
        "Could not locate masters-project repository root."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repository root:", REPO_ROOT)

Repository root: /Users/shivani/Documents/masters-project


In [3]:
from src.evaluation.evaluator import RecommendationEvaluator
from src.policies.hybrid import hybrid_recommendation

DATA_DIR = REPO_ROOT / "data" / "processed" / "google_subset_v1"

# Notebook 04 inputs.
VAL_OCCURRENCES_PATH = DATA_DIR / "val_occurrences.parquet"
TEST_OCCURRENCES_PATH = DATA_DIR / "test_occurrences.parquet"
N04_METADATA_PATH = DATA_DIR / "metadata.json"
N04_VALIDATION_COMPARISON_PATH = DATA_DIR / "baseline_validation_comparison.csv"

# Notebook 05 inputs.
N05_METADATA_PATH = DATA_DIR / "ml_metadata.json"
VAL_ML_PREDICTIONS_PATH = DATA_DIR / "val_ml_predictions.parquet"
TEST_ML_PREDICTIONS_PATH = DATA_DIR / "test_ml_predictions.parquet"

# Notebook 06 outputs.
VALIDATION_POLICY_COMPARISON_PATH = DATA_DIR / "validation_policy_comparison.csv"
VALIDATION_POLICY_INPUTS_PATH = DATA_DIR / "val_policy_inputs.parquet"
VALIDATION_ROW_LEVEL_PATH = DATA_DIR / "validation_policy_row_level.parquet"
N05_FEATURE_MANIFEST_PATH = DATA_DIR / "ml_feature_manifest.json"
VALIDATION_TARGET_PERIOD_SENSITIVITY_PATH = DATA_DIR / "validation_policy_comparison_target_period_sensitivity.csv"
DECISION_MANIFEST_PATH = DATA_DIR / "decision_manifest.json"
FINAL_TEST_COMPARISON_PATH = DATA_DIR / "final_test_policy_comparison.csv"
FINAL_TEST_ROW_LEVEL_PATH = DATA_DIR / "final_test_row_level.parquet"
FINAL_TEST_RUN_METADATA_PATH = DATA_DIR / "final_test_run_metadata.json"

# The final test runs once, only when explicitly enabled.
RUN_FINAL_TEST = False

# Status written to the decision manifest when RUN_FINAL_TEST is False.
# "final" is allowed only after the Google target correction (Task 9) and team approval.
FREEZE_STATUS = "preliminary"
assert FREEZE_STATUS in {"preliminary", "final"}

print("Data directory:", DATA_DIR)
print("RUN_FINAL_TEST:", RUN_FINAL_TEST, "| FREEZE_STATUS:", FREEZE_STATUS)

Data directory: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1
RUN_FINAL_TEST: False | FREEZE_STATUS: preliminary


## Load and Verify N04 and N05 Inputs

N06 consumes the artifacts as produced; it does not rebuild splits, targets, baselines, or predictions.

The test artifact is read here only for identifiers and the P95 baseline. Its target columns are not read until the final-test section.

In [4]:
for required_path in [
    VAL_OCCURRENCES_PATH,
    TEST_OCCURRENCES_PATH,
    N04_METADATA_PATH,
    N04_VALIDATION_COMPARISON_PATH,
    N05_METADATA_PATH,
    VAL_ML_PREDICTIONS_PATH,
    TEST_ML_PREDICTIONS_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required N04/N05 artifact not found: {required_path}. "
            "Run 04_splitAndBaselines.ipynb and 05_evaluateRecommendations.ipynb first."
        )

with open(N04_METADATA_PATH, "r", encoding="utf-8") as file:
    n04_metadata = json.load(file)

with open(N05_METADATA_PATH, "r", encoding="utf-8") as file:
    n05_metadata = json.load(file)

TARGET_CPU_COL = n04_metadata["target_cpu_field"]
TARGET_MEMORY_COL = n04_metadata["target_memory_field"]
SELECTED_MIN_RUNS = int(n04_metadata["selected_min_runs"])
SELECTED_BASELINE_POLICY = str(n04_metadata["selected_baseline_policy"])

# N05 must have produced predictions without scoring the test split.
assert n05_metadata["test_metrics_computed"] is False
assert n05_metadata["hybrid_constructed"] is False
assert n05_metadata["baseline_min_runs"] == SELECTED_MIN_RUNS
assert n05_metadata["n04_selected_baseline_policy"] == SELECTED_BASELINE_POLICY

val_occurrences = pd.read_parquet(VAL_OCCURRENCES_PATH)
val_ml_predictions = pd.read_parquet(VAL_ML_PREDICTIONS_PATH)
test_ml_predictions = pd.read_parquet(TEST_ML_PREDICTIONS_PATH)

# The N05 test prediction file must not carry any test outcome.
outcome_like = [c for c in test_ml_predictions.columns if c.startswith(("actual_", "target_"))]
assert not outcome_like, f"test_ml_predictions.parquet contains outcome columns: {outcome_like}"
n04_validation_comparison = pd.read_csv(N04_VALIDATION_COMPARISON_PATH)

# Test identifiers and the P95 baseline only; no target columns.
TEST_NON_TARGET_COLUMNS = ["occurrence_id", "rec_cpu_p95", "rec_memory_p95"]
test_non_targets = pd.read_parquet(TEST_OCCURRENCES_PATH, columns=TEST_NON_TARGET_COLUMNS)

print("N04 selected baseline:", SELECTED_BASELINE_POLICY, "| min_runs:", SELECTED_MIN_RUNS)
print("N05 selected models:", n05_metadata["selected_models"])
print("Validation occurrences:", len(val_occurrences))
print("Validation ML predictions:", len(val_ml_predictions))
print("Test ML predictions:", len(test_ml_predictions))

N04 selected baseline: P99 | min_runs: 1
N05 selected models: {'cpu': 'random_forest', 'memory': 'random_forest'}
Validation occurrences: 675
Validation ML predictions: 329
Test ML predictions: 731


In [5]:
# Rebuild the common validation population with the exact N04 rule.
COMMON_EVAL_REQUIRED = n04_metadata["common_validation_population"]["required_fields"]
assert COMMON_EVAL_REQUIRED == [TARGET_CPU_COL, TARGET_MEMORY_COL]

val_eval = (
    val_occurrences.loc[val_occurrences[COMMON_EVAL_REQUIRED].notna().all(axis=1)]
    .reset_index(drop=True)
)
assert len(val_eval) == int(n04_metadata["common_validation_population"]["occurrences"])

# N05 validation predictions cover exactly the common population.
assert val_ml_predictions["occurrence_id"].is_unique
assert set(val_ml_predictions["occurrence_id"]) == set(val_eval["occurrence_id"])

val_eval = val_eval.merge(
    val_ml_predictions[
        ["occurrence_id", "actual_cpu", "actual_memory",
         "ml_pred_cpu", "ml_pred_memory", "p95_cpu", "p95_memory"]
    ],
    on="occurrence_id",
    how="left",
    validate="one_to_one",
)

# Actual values and P95 inputs from N05 must equal the N04 columns exactly.
np.testing.assert_array_equal(val_eval["actual_cpu"], val_eval[TARGET_CPU_COL])
np.testing.assert_array_equal(val_eval["actual_memory"], val_eval[TARGET_MEMORY_COL])
np.testing.assert_array_equal(val_eval["p95_cpu"], val_eval["rec_cpu_p95"])
np.testing.assert_array_equal(val_eval["p95_memory"], val_eval["rec_memory_p95"])

# N05 test predictions cover every test occurrence, with the N04 P95 inputs.
assert test_ml_predictions["occurrence_id"].is_unique
assert set(test_ml_predictions["occurrence_id"]) == set(test_non_targets["occurrence_id"])

test_check = test_ml_predictions.merge(
    test_non_targets,
    on="occurrence_id",
    how="left",
    validate="one_to_one",
)
np.testing.assert_array_equal(test_check["p95_cpu"], test_check["rec_cpu_p95"])
np.testing.assert_array_equal(test_check["p95_memory"], test_check["rec_memory_p95"])

for frame, name in [(val_eval, "validation"), (test_ml_predictions, "test")]:
    for column in ["ml_pred_cpu", "ml_pred_memory"]:
        assert np.isfinite(frame[column]).all(), f"{name} {column} must be finite"
        assert (frame[column] >= 0).all(), f"{name} {column} must be non-negative"

print("Common validation occurrences:", len(val_eval))
print("Identifier, target, and P95 alignment checks passed.")

Common validation occurrences: 329
Identifier, target, and P95 alignment checks passed.


## Hybrid Recommendations

The Hybrid rule is the backlog's starting rule: `HybridCPU = max(MLPredictedCPU, P95CPU)` and `HybridMemory = max(MLPredictedMemory, P95Memory)`. It is implemented once in `src/policies/hybrid.py` so that later components (the policy engine, API, and UI) reuse the same rule.

The P95 baseline already falls back to the current request when history is insufficient. When P95 itself is unavailable (missing request and no usable history), Hybrid uses the ML prediction instead of becoming unknown, and the row is flagged.

**Hybrid_P99** applies the same rule with the P99 baseline, the policy N04 selected on validation. It is reported for comparison only.

In [6]:
def add_hybrid_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()

    for resource, rec_prefix in [("cpu", "rec_cpu"), ("memory", "rec_memory")]:
        hybrid, safeguard_missing = hybrid_recommendation(
            result[f"ml_pred_{resource}"],
            result[f"{rec_prefix}_p95"],
        )
        result[f"hybrid_{resource}"] = hybrid
        result[f"hybrid_{resource}_p95_missing"] = safeguard_missing

        hybrid_p99, _ = hybrid_recommendation(
            result[f"ml_pred_{resource}"],
            result[f"{rec_prefix}_p99"],
        )
        result[f"hybrid_p99_{resource}"] = hybrid_p99

    return result


val_eval = add_hybrid_columns(val_eval)

print(
    "Validation rows where Hybrid falls back to ML (P95 unavailable): "
    f"CPU {int(val_eval['hybrid_cpu_p95_missing'].sum())}, "
    f"memory {int(val_eval['hybrid_memory_p95_missing'].sum())}"
)

Validation rows where Hybrid falls back to ML (P95 unavailable): CPU 1, memory 1


## Validation Policy Comparison

All policies are scored by the shared evaluator against the same next-occurrence targets on the same common validation occurrences. The baseline rows must reproduce the N04 validation metrics exactly; this proves N06 evaluates the same thing N04 did.

Valid denominators differ slightly between policies: Request is unknown wherever the request itself is missing (3 rows), and the history baselines are unknown only where both the request and usable history are missing (1 row), so Request has 326 valid any-resource rows and the other baselines 328, while ML and Hybrid are defined on all 329 (Hybrid falls back to ML on the one row without P95). Compare rates together with their `valid_*_count` columns.

The validation ML predictions come from the N05 models trained on training rows only, and N05 chose the model family on these same 329 occurrences, so ML and Hybrid validation numbers are optimistic.

In [7]:
evaluator = RecommendationEvaluator()

POLICIES = {
    "Request": ("rec_cpu_request", "rec_memory_request"),
    "Mean": ("rec_cpu_mean", "rec_memory_mean"),
    "P90": ("rec_cpu_p90", "rec_memory_p90"),
    "P95": ("rec_cpu_p95", "rec_memory_p95"),
    "P99": ("rec_cpu_p99", "rec_memory_p99"),
    "ML": ("ml_pred_cpu", "ml_pred_memory"),
    "Hybrid": ("hybrid_cpu", "hybrid_memory"),
}
SUPPLEMENTARY_POLICIES = {
    "Hybrid_P99": ("hybrid_p99_cpu", "hybrid_p99_memory"),
}
BASELINE_POLICY_NAMES = ["Request", "Mean", "P90", "P95", "P99"]


def evaluate_policies(df: pd.DataFrame, policies: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    summaries = []
    row_level = []

    for policy_name, (cpu_col, memory_col) in policies.items():
        evaluated = evaluator.evaluate(
            data=df,
            actual_cpu_col=TARGET_CPU_COL,
            actual_memory_col=TARGET_MEMORY_COL,
            recommended_cpu_col=cpu_col,
            recommended_memory_col=memory_col,
            policy_name=policy_name,
            id_cols=["occurrence_id", "recurrence_key"],
        )
        summaries.append({"policy": policy_name, **evaluator.summarize(evaluated)})
        row_level.append(
            evaluated[
                ["occurrence_id", "recurrence_key", "policy_name",
                 TARGET_CPU_COL, TARGET_MEMORY_COL, cpu_col, memory_col,
                 "cpu_violation", "memory_violation", "any_resource_violation",
                 "cpu_utilization", "memory_utilization", "cpu_waste", "memory_waste",
                 "cpu_shortfall", "memory_shortfall"]
            ].rename(columns={cpu_col: "recommended_cpu", memory_col: "recommended_memory"})
        )

    return pd.DataFrame(summaries), pd.concat(row_level, ignore_index=True)


validation_summary, validation_row_level = evaluate_policies(val_eval, {**POLICIES, **SUPPLEMENTARY_POLICIES})
validation_summary.insert(1, "supplementary", validation_summary["policy"].isin(SUPPLEMENTARY_POLICIES))
validation_summary.insert(2, "n_common_occurrences", len(val_eval))

# The baselines must reproduce N04 exactly.
n04_rows = n04_validation_comparison[
    n04_validation_comparison["min_runs"] == SELECTED_MIN_RUNS
].set_index("policy")

for policy_name in BASELINE_POLICY_NAMES:
    row = validation_summary.set_index("policy").loc[policy_name]
    for metric in [column for column in n04_rows.columns if column in row.index and column not in {"policy", "min_runs"}]:
        if not np.isclose(float(row[metric]), float(n04_rows.loc[policy_name, metric]), rtol=1e-9, atol=1e-12, equal_nan=True):
            raise RuntimeError(f"{policy_name} {metric}: N06={row[metric]} but N04={n04_rows.loc[policy_name, metric]}")

assert (validation_summary["row_count"] == len(val_eval)).all()

print("All N04 baseline validation metrics reproduced exactly.")

DISPLAY_COLUMNS = [
    "policy", "supplementary",
    "valid_any_resource_violation_count", "any_resource_violation_pct",
    "cpu_violation_pct", "memory_violation_pct",
    "average_cpu_utilization", "average_memory_utilization",
    "average_cpu_waste", "average_memory_waste",
    "average_cpu_shortfall", "average_memory_shortfall",
]

display(validation_summary[DISPLAY_COLUMNS].round(4))

All N04 baseline validation metrics reproduced exactly.


,policy,supplementary,valid_any_resource_violation_count,any_resource_violation_pct,cpu_violation_pct,memory_violation_pct,average_cpu_utilization,average_memory_utilization,average_cpu_waste,average_memory_waste,average_cpu_shortfall,average_memory_shortfall
0,Request,False,326,55.5215,52.1472,38.0368,0.6227,0.5297,0.9570,0.4490,2.4214,0.3021
1,Mean,False,328,42.9878,35.3659,34.1463,0.4907,0.5091,1.7405,0.3628,1.8869,0.2167
2,P90,False,328,28.6585,22.2561,21.6463,0.3866,0.4061,4.8557,0.9805,1.3347,0.1422
3,P95,False,328,25.3049,18.5976,19.8171,0.3411,0.3706,8.4491,1.8646,1.0636,0.1030
4,P99,False,328,21.9512,16.7683,17.0732,0.2979,0.3371,19.0411,3.3459,0.9722,0.0862
5,ML,False,329,34.9544,30.3951,25.2280,0.4583,0.4246,0.4622,0.1554,2.3614,0.2984
6,Hybrid,False,329,17.9331,13.3739,14.5897,0.2857,0.2969,8.4696,1.8673,1.0427,0.1204
7,Hybrid_P99,True,329,15.5015,11.8541,12.7660,0.2465,0.2704,19.0258,3.3433,0.9521,0.1038


In [8]:
validation_summary.insert(3, "result_status", "preliminary")
validation_summary.to_csv(VALIDATION_POLICY_COMPARISON_PATH, index=False)
validation_row_level.to_parquet(VALIDATION_ROW_LEVEL_PATH, index=False)

saved_row_level = pd.read_parquet(VALIDATION_ROW_LEVEL_PATH)
pd.testing.assert_frame_equal(saved_row_level, validation_row_level)
assert len(saved_row_level) == len(val_eval) * (len(POLICIES) + len(SUPPLEMENTARY_POLICIES))
print("Saved:", VALIDATION_ROW_LEVEL_PATH, "| rows:", len(saved_row_level))

saved_validation = pd.read_csv(VALIDATION_POLICY_COMPARISON_PATH)
assert list(saved_validation["policy"]) == list(validation_summary["policy"])
assert len(saved_validation) == len(validation_summary)

print("Saved:", VALIDATION_POLICY_COMPARISON_PATH)

Saved: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1/validation_policy_row_level.parquet | rows: 2632
Saved: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1/validation_policy_comparison.csv


### Standard policy input columns

The validation inputs to every policy are saved with the standard column names used across the project: `actual_*`, `ml_pred_*`, `p95_*`, `hybrid_*` (plus the remaining baselines and the Hybrid fallback flags), keyed by `occurrence_id`.

In [9]:
policy_inputs = pd.DataFrame({
    "occurrence_id": val_eval["occurrence_id"],
    "occurrence_key": val_eval["occurrence_key"],
    "recurrence_key": val_eval["recurrence_key"],
    "execution_start_time_us": val_eval["execution_start_time_us"],
    "target_next_occurrence_start_time_us": val_eval["target_next_occurrence_start_time_us"],
    "actual_cpu": val_eval[TARGET_CPU_COL],
    "actual_memory": val_eval[TARGET_MEMORY_COL],
    "request_cpu": val_eval["rec_cpu_request"],
    "request_memory": val_eval["rec_memory_request"],
    "mean_cpu": val_eval["rec_cpu_mean"],
    "mean_memory": val_eval["rec_memory_mean"],
    "p90_cpu": val_eval["rec_cpu_p90"],
    "p90_memory": val_eval["rec_memory_p90"],
    "p95_cpu": val_eval["rec_cpu_p95"],
    "p95_memory": val_eval["rec_memory_p95"],
    "p99_cpu": val_eval["rec_cpu_p99"],
    "p99_memory": val_eval["rec_memory_p99"],
    "ml_pred_cpu": val_eval["ml_pred_cpu"],
    "ml_pred_memory": val_eval["ml_pred_memory"],
    "hybrid_cpu": val_eval["hybrid_cpu"],
    "hybrid_memory": val_eval["hybrid_memory"],
    "hybrid_cpu_p95_missing": val_eval["hybrid_cpu_p95_missing"],
    "hybrid_memory_p95_missing": val_eval["hybrid_memory_p95_missing"],
})
policy_inputs.to_parquet(VALIDATION_POLICY_INPUTS_PATH, index=False)

reloaded_inputs = pd.read_parquet(VALIDATION_POLICY_INPUTS_PATH)
pd.testing.assert_frame_equal(reloaded_inputs, policy_inputs)
assert reloaded_inputs["occurrence_id"].is_unique
assert len(reloaded_inputs) == len(val_eval)

print("Saved:", VALIDATION_POLICY_INPUTS_PATH, "| rows:", len(reloaded_inputs))

Saved: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1/val_policy_inputs.parquet | rows: 329


### Sensitivity: validation targets inside the validation period only

N04 builds next-occurrence targets before splitting, so some validation targets fall in the test period. This table re-scores the same policies on the validation occurrences whose target starts on or before the validation boundary. It is reported for transparency only; no decision uses it.

In [10]:
VAL_END_US = int(n04_metadata["split_boundaries"]["val_end"])
target_in_test_period = val_eval["target_next_occurrence_start_time_us"] > VAL_END_US

print("Validation occurrences with targets in the test period:", int(target_in_test_period.sum()), "of", len(val_eval))

sensitivity_summary, _ = evaluate_policies(
    val_eval.loc[~target_in_test_period].reset_index(drop=True),
    {**POLICIES, **SUPPLEMENTARY_POLICIES},
)
sensitivity_summary.insert(1, "population", "targets_within_validation_period")
sensitivity_summary.insert(2, "n_common_occurrences", int((~target_in_test_period).sum()))
sensitivity_summary.insert(3, "result_status", "preliminary")
sensitivity_summary.to_csv(VALIDATION_TARGET_PERIOD_SENSITIVITY_PATH, index=False)

display(sensitivity_summary[["policy", "n_common_occurrences", "valid_any_resource_violation_count",
                             "any_resource_violation_pct", "cpu_violation_pct", "memory_violation_pct",
                             "average_cpu_waste", "average_memory_waste"]].round(4))
print("Saved:", VALIDATION_TARGET_PERIOD_SENSITIVITY_PATH)

Validation occurrences with targets in the test period: 104 of 329


,policy,n_common_occurrences,valid_any_resource_violation_count,any_resource_violation_pct,cpu_violation_pct,memory_violation_pct,average_cpu_waste,average_memory_waste
0,Request,225,222,57.6577,52.7027,40.5405,0.9145,0.4588
1,Mean,225,224,40.1786,31.2500,35.2679,2.0858,0.4444
2,P90,225,224,26.7857,20.0893,20.9821,5.9711,1.2491
3,P95,225,224,23.2143,15.6250,19.1964,10.7431,2.4617
4,P99,225,224,18.3036,12.9464,15.6250,25.0399,4.4940
5,ML,225,225,36.8889,32.4444,26.2222,0.5050,0.1738
6,Hybrid,225,225,17.7778,12.8889,14.2222,10.7340,2.4578
7,Hybrid_P99,225,225,14.2222,10.6667,12.0000,24.9658,4.4802


Saved: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1/validation_policy_comparison_target_period_sensitivity.csv


## Decision Freeze (Decision Manifest)

Every decision needed to produce the final recommendations is written to `decision_manifest.json`, together with a SHA-256 hash of every input artifact and of the shared evaluator and Hybrid rule. The manifest also records the commit and a SHA-256 of this notebook's code (with only the `RUN_FINAL_TEST` and `FREEZE_STATUS` values blanked). The final-test section refuses to run unless the manifest is final, every hashed input still matches, the current commit equals the frozen commit, the only uncommitted change is this notebook, and its code hash still matches.

In [11]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def git_state() -> dict:
    def run(*args):
        result = subprocess.run(["git", *args], cwd=REPO_ROOT, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f"git {' '.join(args)} failed: {result.stderr.strip()}")
        return result.stdout.strip()

    return {"commit": run("rev-parse", "HEAD"), "uncommitted_changes": bool(run("status", "--porcelain"))}


NOTEBOOK_PATH = REPO_ROOT / "pipelines" / "google_subset" / "06_finalPolicyComparison.ipynb"


def notebook_code_sha256(path: Path) -> str:
    # Hash of the saved notebook's code cells; only the two run parameters are blanked.
    with open(path, "r", encoding="utf-8") as file:
        notebook = json.load(file)
    digest = hashlib.sha256()
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        source = "".join(cell["source"])
        lines = [
            line.split("=")[0] + "= <parameter>"
            if line.startswith(("RUN_FINAL_TEST =", "FREEZE_STATUS ="))
            else line
            for line in source.splitlines()
        ]
        digest.update("\n".join(lines).encode("utf-8"))
        digest.update(b"\x00")
    return digest.hexdigest()


def uncommitted_paths() -> set:
    result = subprocess.run(["git", "status", "--porcelain"], cwd=REPO_ROOT, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"git status failed: {result.stderr.strip()}")
    return {line[3:].strip() for line in result.stdout.splitlines() if line.strip()}


locked_model_paths = sorted((DATA_DIR / "n05_locked_models").glob("*.joblib"))
assert len(locked_model_paths) == 2, locked_model_paths

HASHED_INPUTS = [
    VAL_OCCURRENCES_PATH,
    TEST_OCCURRENCES_PATH,
    N04_METADATA_PATH,
    N04_VALIDATION_COMPARISON_PATH,
    N05_METADATA_PATH,
    N05_FEATURE_MANIFEST_PATH,
    VAL_ML_PREDICTIONS_PATH,
    TEST_ML_PREDICTIONS_PATH,
    *locked_model_paths,
    REPO_ROOT / "src" / "evaluation" / "evaluator.py",
    REPO_ROOT / "src" / "evaluation" / "metrics.py",
    REPO_ROOT / "src" / "policies" / "hybrid.py",
]

decision_manifest = {
    "status": FREEZE_STATUS,
    "status_reason": (
        "Google targets are not yet split-isolated (data_integration_tasks.md Task 9); set to final only after Task 9 and team approval."
        if FREEZE_STATUS == "preliminary"
        else "Frozen for the one-time final-test evaluation."
    ),
    "dataset": n04_metadata["dataset"],
    "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "notebook": "06_finalPolicyComparison.ipynb",
    "code_state": git_state(),
    "notebook_code_sha256": notebook_code_sha256(NOTEBOOK_PATH),
    "decisions": {
        "target_cpu_field": TARGET_CPU_COL,
        "target_memory_field": TARGET_MEMORY_COL,
        "baseline_min_runs": SELECTED_MIN_RUNS,
        "selected_baseline_policy": SELECTED_BASELINE_POLICY,
        "baseline_selection_source": "Notebook 04 metadata.json (validation only)",
        "ml_selected_models": n05_metadata["selected_models"],
        "ml_locked_params": n05_metadata["locked_params"],
        "ml_feature_manifest": n05_metadata["feature_manifest"],
        "hybrid_rule": "hybrid = max(ML prediction, P95 baseline) per resource",
        "hybrid_fallback": "ML prediction where the P95 baseline is unavailable",
        "hybrid_rule_rationale": "Backlog starting rule (New Issue 6 / Issue 07); fixed in advance, not tuned on validation",
        "supplementary_policies": {"Hybrid_P99": "max(ML prediction, P99 baseline); reported, not a decision"},
        "core_policies": list(POLICIES),
        "evaluation_population": "occurrences with both next-occurrence CPU and memory targets (N04 rule)",
        "evaluator": "src/evaluation/evaluator.py RecommendationEvaluator",
    },
    "input_sha256": {
        str(path.relative_to(REPO_ROOT)): sha256_file(path)
        for path in HASHED_INPUTS
    },
    "versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}

if RUN_FINAL_TEST:
    # The final test only reads the manifest frozen in an earlier run.
    print("RUN_FINAL_TEST is True: the existing decision manifest is not rewritten.")
else:
    if DECISION_MANIFEST_PATH.exists():
        with open(DECISION_MANIFEST_PATH, "r", encoding="utf-8") as file:
            existing_status = json.load(file).get("status")
        if existing_status == "final":
            raise RuntimeError("A final decision manifest already exists and must not be overwritten.")

    if FREEZE_STATUS == "final" and decision_manifest["code_state"]["uncommitted_changes"]:
        raise RuntimeError("A final decision manifest requires a clean, committed working tree.")

    with open(DECISION_MANIFEST_PATH, "w", encoding="utf-8") as file:
        json.dump(decision_manifest, file, indent=2)

    with open(DECISION_MANIFEST_PATH, "r", encoding="utf-8") as file:
        assert json.load(file)["input_sha256"] == decision_manifest["input_sha256"]

    print("Saved:", DECISION_MANIFEST_PATH, "| status:", FREEZE_STATUS)
    print("Hashed inputs:", len(decision_manifest["input_sha256"]))
    print("Code state:", decision_manifest["code_state"])

Saved: /Users/shivani/Documents/masters-project/data/processed/google_subset_v1/decision_manifest.json | status: preliminary
Hashed inputs: 13
Code state: {'commit': 'c6869db17f7be6053833330d0f97828744d59f03', 'uncommitted_changes': True}


## One-Time Final-Test Evaluation

This section runs only when `RUN_FINAL_TEST = True`. Before reading any test target it:

1. requires the decision manifest and verifies that every hashed input is unchanged;
2. refuses to run if a final-test result already exists, so the test is evaluated once.

It then scores the seven core policies on the test occurrences with both targets present, through the same shared evaluator, and saves the policy-level comparison, the row-level results, and run metadata. `final_test_policy_comparison.csv` is the only source for final Google report numbers. No decision may be changed afterwards based on these results; if a true implementation bug is found, the invalid run is documented and the lock/test process restarted transparently.

In [12]:
if not RUN_FINAL_TEST:
    print("Final-test evaluation is disabled (RUN_FINAL_TEST = False). No test target was read.")
else:
    # Gate 1: the decision manifest must exist, be final, and every input must be unchanged.
    if not DECISION_MANIFEST_PATH.exists():
        raise RuntimeError("Decision manifest missing; final-test evaluation is not allowed.")

    with open(DECISION_MANIFEST_PATH, "r", encoding="utf-8") as file:
        frozen = json.load(file)

    if frozen.get("status") != "final":
        raise RuntimeError(
            f"Decision manifest status is {frozen.get('status')!r}; the final test requires status 'final'."
        )

    if frozen["code_state"]["uncommitted_changes"] or not frozen["code_state"]["commit"]:
        raise RuntimeError("The decision manifest was not frozen from a clean, committed tree.")

    if git_state()["commit"] != frozen["code_state"]["commit"]:
        raise RuntimeError("The current commit differs from the commit frozen in the decision manifest.")

    notebook_relative = str(NOTEBOOK_PATH.relative_to(REPO_ROOT))
    other_changes = sorted(uncommitted_paths() - {notebook_relative})
    if other_changes:
        raise RuntimeError(f"Uncommitted changes other than this notebook: {other_changes}")

    if notebook_code_sha256(NOTEBOOK_PATH) != frozen["notebook_code_sha256"]:
        raise RuntimeError("This notebook's code changed after the decision freeze.")

    changed = [
        path for path, digest in frozen["input_sha256"].items()
        if sha256_file(REPO_ROOT / path) != digest
    ]
    if changed:
        raise RuntimeError(f"Inputs changed after the decision freeze: {changed}")

    # Gate 2: the final test is evaluated once.
    if any(path.exists() for path in [FINAL_TEST_COMPARISON_PATH, FINAL_TEST_ROW_LEVEL_PATH, FINAL_TEST_RUN_METADATA_PATH]):
        raise RuntimeError(
            "A final-test result already exists. Document the previous run before any rerun."
        )

    test_occurrences = pd.read_parquet(TEST_OCCURRENCES_PATH)
    test_eval = (
        test_occurrences.loc[test_occurrences[COMMON_EVAL_REQUIRED].notna().all(axis=1)]
        .reset_index(drop=True)
        .merge(
            test_ml_predictions[["occurrence_id", "ml_pred_cpu", "ml_pred_memory"]],
            on="occurrence_id",
            how="left",
            validate="one_to_one",
        )
    )
    test_eval = add_hybrid_columns(test_eval)

    test_summary, test_row_level = evaluate_policies(test_eval, POLICIES)
    test_summary.insert(1, "n_common_occurrences", len(test_eval))
    assert (test_summary["row_count"] == len(test_eval)).all()

    test_summary.to_csv(FINAL_TEST_COMPARISON_PATH, index=False)
    test_row_level.to_parquet(FINAL_TEST_ROW_LEVEL_PATH, index=False)

    run_metadata = {
        "run_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "decision_manifest_sha256": sha256_file(DECISION_MANIFEST_PATH),
        "code_state": git_state(),
        "test_occurrences": len(test_occurrences),
        "evaluated_occurrences": len(test_eval),
        "policies": list(POLICIES),
        "outputs": {
            "policy_comparison": FINAL_TEST_COMPARISON_PATH.name,
            "row_level": FINAL_TEST_ROW_LEVEL_PATH.name,
        },
    }
    with open(FINAL_TEST_RUN_METADATA_PATH, "w", encoding="utf-8") as file:
        json.dump(run_metadata, file, indent=2)

    # Reload verification.
    assert len(pd.read_csv(FINAL_TEST_COMPARISON_PATH)) == len(test_summary)
    saved_rows = pd.read_parquet(FINAL_TEST_ROW_LEVEL_PATH)
    assert len(saved_rows) == len(test_eval) * len(POLICIES)
    assert set(saved_rows["occurrence_id"]) == set(test_eval["occurrence_id"])

    print("Final-test evaluation complete:", len(test_eval), "occurrences,", len(POLICIES), "policies.")
    display(test_summary[[c for c in DISPLAY_COLUMNS if c != "supplementary"]].round(4))

Final-test evaluation is disabled (RUN_FINAL_TEST = False). No test target was read.


# N06 Handoff

After this notebook runs:

- N04 and N05 artifacts were consumed without rebuilding splits, targets, baselines, or predictions, and their identifiers, targets, and P95 inputs were verified;
- Hybrid recommendations were built with the shared rule in `src/policies/hybrid.py`;
- Request, Mean, P90, P95, P99, ML-only, and Hybrid were compared on the common validation population through the shared evaluator, with the N04 baseline metrics reproduced exactly (`validation_policy_comparison.csv`);
- row-level validation evaluator output for every policy was saved in `validation_policy_row_level.parquet`;
- the validation inputs of every policy were saved with standard columns (`actual_*`, `ml_pred_*`, `p95_*`, `hybrid_*`) in `val_policy_inputs.parquet`;
- a sensitivity table re-scored the policies on validation occurrences whose targets stay inside the validation period;
- every decision and a hash of every input were recorded in `decision_manifest.json` with `"status": "preliminary"`;
- all results here are **preliminary** until the Google target correction (Task 9). The final-test section runs only if enabled and only when the manifest status is `"final"` and every input hash matches; when it runs, `final_test_policy_comparison.csv` is the only source for final Google report numbers.